In [2]:
"""
pdf_text_extraction.py

Extracts plain text from each PDF using DocLayout-YOLO for correct
reading order (title + body text only, tables and figures are skipped
entirely, not needed for this pipeline).

Saves ONE .txt file per paper into PDF_extracted_txt/, named by its
DOI in the project's safe form (matching the Elsevier .txt naming
convention: replace '.' with '-' and '/' with '_'), so both sources
look identical going into the chunking step.

This script does NOT chunk anything. That is a separate step
(pdf_chunk_processing.py) on purpose: the raw extracted text can now
be opened directly in Notepad and checked for reading-order or
fragmentation problems, before it's ever cut into chunks. Re-running
the chunking logic after a chunk_utils.py fix also no longer requires
re-running the slow DocLayout-YOLO step.

Crash resilience: pdf_extraction_log.csv tracks which source PDFs have
already been extracted (by original filename, since the DOI isn't
known until after extraction runs). Rerunning the script skips any
PDF already logged, so an interrupted run resumes cleanly.
"""

import csv
import os
import re

import fitz  # PyMuPDF
from PIL import Image

PDF_FOLDER = r"c:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\DOWNLOAD ARTICLES\PDF_files"
OUTPUT_FOLDER = "PDF_extracted_txt"
LOG_PATH = "pdf_extraction_log.csv"

MODEL_REPO = "juliozhao/DocLayout-YOLO-DocStructBench"
MODEL_FILE = "doclayout_yolo_docstructbench_imgsz1024.pt"

CONF_THRESHOLD = 0.25
RENDER_DPI = 200

# Only title and plain text regions are kept. Tables, figures, captions,
# footnotes, and formulas are skipped entirely.
TEXT_CLASSES = {0, 1}  # 0: title, 1: plain_text

DOI_PATTERN = re.compile(
    r"(?:doi:\s*|https?://doi\.org/)?(10\.\d{4,9}/[-._;()/:A-Za-z0-9]+)",
    re.IGNORECASE,
)


def doi_to_safe_id(doi: str) -> str:
    return doi.strip().replace(".", "-").replace("/", "_")


def load_model():
    from doclayout_yolo import YOLOv10
    from huggingface_hub import hf_hub_download
    print("Loading DocLayout-YOLO model (first run downloads ~60MB)...")
    model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
    return YOLOv10(model_path)


def render_page(page, dpi=RENDER_DPI):
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)


def extract_text_from_box(page, box, dpi=RENDER_DPI):
    scale = 72 / dpi
    rect = fitz.Rect(box[0] * scale, box[1] * scale, box[2] * scale, box[3] * scale)
    return page.get_text("text", clip=rect).strip()


def extract_doi(text: str) -> str:
    """Search only the first ~3000 characters for the paper's own DOI.

    A paper's own DOI always appears in the masthead/header near the
    top of page 1. Searching the full text risks matching a citation's
    DOI instead if the paper's own DOI wasn't captured during layout
    detection, confirmed to happen on at least one real PDF, where the
    only DOI found anywhere was a 1985 reference buried in the
    bibliography, and that wrong DOI silently became the file's name.
    """
    header_region = text[:3000].replace("\n", " ")
    match = DOI_PATTERN.search(header_region)
    return match.group(1).strip() if match else None


import re as _re

# Matches a Conclusion header the same way chunk_utils.py does: short
# standalone line, optional leading number ("4.", "4)"), case-insensitive.
_CONCLUSION_HEADER_PREFIX = _re.compile(r"^\d+(\.\d+)*[\.\)]?\s*")
_CONCLUSION_PATTERN = _re.compile(r"conclusion[s]?", _re.IGNORECASE)
_DOI_MARKER_PATTERN = _re.compile(r"doi\s*:\s*10\.|doi\.org/10\.", _re.IGNORECASE)


def _is_conclusion_header(text: str) -> bool:
    stripped = text.strip().rstrip(":").strip()
    candidate = _CONCLUSION_HEADER_PREFIX.sub("", stripped).strip()
    return 0 < len(candidate) < 60 and bool(_CONCLUSION_PATTERN.fullmatch(candidate))


def _looks_like_reference_entry(text: str) -> bool:
    """Catches reference-list content with no reliable header to trigger
    on at all (confirmed to happen: one real PDF's Conclusion section,
    header included, never appeared anywhere in the extracted text, so
    the header-based stop above had nothing to detect). Every reference
    entry seen across every PDF checked so far contains a literal
    'doi:' marker, which essentially never appears in ordinary body
    text, this is a second, independent stop trigger.

    Requires a minimum length before counting a match. Confirmed on a
    real PDF: some publishers (Taylor & Francis) print their own DOI
    twice in the front matter, once inside a long citation block and
    again as a short bare link line ("To link to this article:
    https://doi.org/..."). Both matched the pattern and, being right
    next to each other, falsely triggered the density check below. A
    real bibliography entry is always a full citation (author names,
    title, journal, year), never just a bare link, so filtering out
    short matches removes the false positive without affecting real
    reference entries.
    """
    if len(text.strip()) < 80:
        return False
    return bool(_DOI_MARKER_PATTERN.search(text))


def extract_pdf_text(pdf_path, model) -> str:
    """Return the ordered plain-text content of one PDF (title + body only),
    stopping the instant a Conclusion header is detected.

    An earlier version tried to finish extracting the rest of that page
    before stopping, to capture the conclusion paragraph itself. That
    failed on real PDFs where the reference list appears immediately
    after the Conclusion header, on the same page, before the actual
    conclusion text (confirmed directly: "Conclusion" at one line,
    reference citations starting within the next few lines). "Finish
    the page" ended up capturing that leaked reference block too.

    The only reliably safe rule is to stop capturing text the moment
    the Conclusion header itself is seen, nothing from that point
    forward is trusted, including the header itself. This does mean
    the conclusion paragraph is not captured, references were the
    priority to eliminate, and conclusions largely restate findings
    already present in Results/Discussion.

    Includes the same deduplication guard as before, for any repeats
    that occur before the Conclusion is reached.
    """
    doc = fitz.open(pdf_path)
    all_text = []
    seen_blocks = set()
    # doi_hit_positions tracks the index (within all_text) of every block
    # that matched a doi: pattern, keeping only hits within the last
    # DOI_HIT_WINDOW appended blocks. A single isolated hit is a
    # self-citation (every publisher prints one somewhere in front
    # matter, MDPI specifically prints it AFTER the abstract rather than
    # before the title, confirmed directly: three real PDFs were cut off
    # right at the abstract because a fixed character threshold alone
    # couldn't tell a lone self-citation from the start of a real
    # reference list). A real bibliography has many doi: entries packed
    # closely together, that density, not position, is the reliable
    # signal. Once DOI_HIT_COUNT_TO_TRIGGER hits land within the window,
    # roll all_text back to just before the first of them and stop.
    doi_hit_positions = []
    DOI_HIT_WINDOW = 15       # blocks
    DOI_HIT_COUNT_TO_TRIGGER = 2

    for page_num in range(len(doc)):
        page = doc[page_num]
        page_image = render_page(page)

        results = model.predict(page_image, imgsz=1024, conf=CONF_THRESHOLD, verbose=False)
        if not results or results[0].boxes is None:
            continue

        boxes = results[0].boxes.xyxy.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy().astype(int)

        order = sorted(range(len(boxes)), key=lambda i: (round(boxes[i][1] / 50) * 50, boxes[i][0]))

        stop_now = False
        for i in order:
            if classes[i] not in TEXT_CLASSES:
                continue
            text = extract_text_from_box(page, boxes[i])
            if not text:
                continue

            if _is_conclusion_header(text):
                stop_now = True
                break  # do not append this block or anything after it

            block_key = text.strip()
            if len(block_key) > 40 and block_key in seen_blocks:
                continue
            if len(block_key) > 40:
                seen_blocks.add(block_key)

            all_text.append(text)

            if _looks_like_reference_entry(text):
                doi_hit_positions.append(len(all_text) - 1)
                doi_hit_positions = [p for p in doi_hit_positions if p > len(all_text) - 1 - DOI_HIT_WINDOW]
                if len(doi_hit_positions) >= DOI_HIT_COUNT_TO_TRIGGER:
                    all_text = all_text[:doi_hit_positions[0]]  # discard from the first recent hit onward
                    stop_now = True
                    break

        if stop_now:
            break

    doc.close()
    return "\n\n".join(all_text)


# ---------------------------------------------------------------------------
# Crash-resilience log (tracks by source PDF filename, not DOI, since DOI
# is only known after extraction runs)
# ---------------------------------------------------------------------------
def load_processed_log(log_path: str) -> set:
    done = set()
    if not os.path.exists(log_path):
        return done
    with open(log_path, "r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            done.add(row["source_file"])
    return done


def append_log(log_path: str, source_file: str, output_file: str, doi: str):
    is_new = not os.path.exists(log_path)
    with open(log_path, "a", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        if is_new:
            writer.writerow(["source_file", "output_file", "doi"])
        writer.writerow([source_file, output_file, doi])


def run():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    # Below this, a paper almost certainly got cut off before reaching
    # real Methods/Results content, based on every case actually seen:
    # successful extractions ranged 19,000-64,000 characters, every
    # truncation bug produced under 2,500. Set well above that gap.
    MIN_CHARS_LIKELY_OK = 3000
    FLAGGED_LOG_PATH = "pdf_extraction_flagged.csv"

    already_done = load_processed_log(LOG_PATH)
    if already_done:
        print(f"Resuming: {len(already_done)} PDFs already extracted, will be skipped.")

    pdf_files = [
        f for f in sorted(os.listdir(PDF_FOLDER))
        if f.lower().endswith(".pdf") and f not in already_done
    ]
    print(f"Found {len(pdf_files)} PDFs left to extract in {PDF_FOLDER}/")

    if not pdf_files:
        print("Nothing new to extract.")
        return

    model = load_model()
    failed = []
    flagged = []  # (filename, safe_id, char_count) for anything suspiciously short
    ok_count = 0

    for i, fname in enumerate(pdf_files, 1):
        pdf_path = os.path.join(PDF_FOLDER, fname)
        print(f"[{i}/{len(pdf_files)}] {fname}")
        try:
            full_text = extract_pdf_text(pdf_path, model)
            doi = extract_doi(full_text)
            safe_id = doi_to_safe_id(doi) if doi else os.path.splitext(fname)[0]

            output_path = os.path.join(OUTPUT_FOLDER, f"{safe_id}.txt")
            with open(output_path, "w", encoding="utf-8") as out:
                out.write(full_text)

            append_log(LOG_PATH, fname, f"{safe_id}.txt", doi or "")
            note = doi if doi else "(DOI not found, used PDF filename instead)"
            char_count = len(full_text)

            if char_count < MIN_CHARS_LIKELY_OK:
                flagged.append((fname, safe_id, char_count))
                print(f"    doi={note}  -> {safe_id}.txt  ({char_count} chars, FLAGGED: unusually short)")
            else:
                ok_count += 1
                print(f"    doi={note}  -> {safe_id}.txt  ({char_count} chars)")
        except Exception as e:
            print(f"    [ERROR] {e}")
            failed.append(fname)

    total_this_run = len(pdf_files)
    print()
    print("=" * 60)
    print(f"Extraction summary for this run:")
    print(f"  Total PDFs processed:  {total_this_run}")
    print(f"  Likely extracted OK:   {ok_count}")
    print(f"  Flagged (short, needs review): {len(flagged)}")
    print(f"  Failed with an error:  {len(failed)}")
    print("=" * 60)

    if flagged:
        is_new = not os.path.exists(FLAGGED_LOG_PATH)
        with open(FLAGGED_LOG_PATH, "a", encoding="utf-8", newline="") as f:
            writer = csv.writer(f)
            if is_new:
                writer.writerow(["source_file", "output_file", "char_count"])
            for fname, safe_id, char_count in flagged:
                writer.writerow([fname, f"{safe_id}.txt", char_count])
        print(f"Flagged files written to {FLAGGED_LOG_PATH}, review these specifically.")

    if failed:
        print(f"Failed files: {failed}")


if __name__ == "__main__":
    run()

Found 82 PDFs left to extract in c:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\DOWNLOAD ARTICLES\PDF_files/
Loading DocLayout-YOLO model (first run downloads ~60MB)...


[1/82] 10-1002_cche-10383.pdf
    doi=(DOI not found, used PDF filename instead)  -> 10-1002_cche-10383.txt  (36763 chars)
[2/82] 10-1002_cche-10589.pdf
    doi=(DOI not found, used PDF filename instead)  -> 10-1002_cche-10589.txt  (47138 chars)
[3/82] 10-1002_food-200390094.pdf
    doi=(DOI not found, used PDF filename instead)  -> 10-1002_food-200390094.txt  (20805 chars)
[4/82] 10-1002_fsn3-4511.pdf
    doi=(DOI not found, used PDF filename instead)  -> 10-1002_fsn3-4511.txt  (79272 chars)
[5/82] 10-1002_jsfa-12702.pdf
    doi=(DOI not found, used PDF filename instead)  -> 10-1002_jsfa-12702.txt  (32967 chars)
[6/82] 10-1002_jsfa-13233.pdf
    doi=(DOI not found, used PDF filename instead)  -> 10-1002_jsfa-13233.txt  (38512 chars)
[7/82] 10-1002_jsfa-2334.pdf
    doi=(DOI not found, used PDF filename instead)  -> 10-1002_jsfa-2334.txt  (14525 chars)
[8/82] 10-1002_jsfa-70829.pdf
    doi=(DOI not found, used PDF filename instead)  -> 10-1002_jsfa-70829.txt  (46276 chars)
[9/82] 10-10